In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_mlp_mixer_utkface_age
from train import WarmUpCosine, AdamW, LastNSaver, make_train_ds, make_test_ds

In [4]:
num_classes = 3
initial_lr = 5e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
model = build_mlp_mixer_utkface_age(num_classes=4)

In [7]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)

In [8]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [9]:
saver = LastNSaver(n=20)

In [10]:
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.2   # 0.2~0.4 都常用；过拟合明显就加大
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 17s - loss: 1.2701 - accuracy: 0.4032 - val_loss: 1.7591 - val_accuracy: 0.2573 - 17s/epoch - 107ms/step
Epoch 2/200
156/156 - 9s - loss: 1.0999 - accuracy: 0.5339 - val_loss: 1.7645 - val_accuracy: 0.3228 - 9s/epoch - 56ms/step
Epoch 3/200
156/156 - 9s - loss: 1.0208 - accuracy: 0.5894 - val_loss: 0.9435 - val_accuracy: 0.6087 - 9s/epoch - 55ms/step
Epoch 4/200
156/156 - 9s - loss: 0.9600 - accuracy: 0.6326 - val_loss: 0.8576 - val_accuracy: 0.6620 - 9s/epoch - 55ms/step
Epoch 5/200
156/156 - 9s - loss: 0.9134 - accuracy: 0.6591 - val_loss: 0.6950 - val_accuracy: 0.7350 - 9s/epoch - 55ms/step
Epoch 6/200
156/156 - 9s - loss: 0.8759 - accuracy: 0.6807 - val_loss: 0.7267 - val_accuracy: 0.7155 - 9s/epoch - 55ms/step
Epoch 7/200
156/156 - 9s - loss: 0.8290 - accuracy: 0.7067 - val_loss: 0.8755 - val_accuracy: 0.6335 - 9s/epoch - 55ms/step
Epoch 8/200
156/156 - 9s - loss: 0.7834 - accuracy: 0.7281 - val_loss: 0.5832 - val_accuracy: 0.7735 - 9s/epoch - 55ms/step
Epoch

In [11]:
model.save("Mix_8.h5")